# Micrograd: A Tiny Automatic Differentiation Engine

Based on Andrej Karpathy's educational implementation.

This notebook covers:
- Automatic differentiation (backpropagation)
- Building neural networks from scratch
- Training with gradient descent
- Simple classification example

## Part 1: The Value Class

Core building block for automatic differentiation.

In [7]:
import math
from typing import Set, Callable

class Value:
    """Scalar value with automatic differentiation support."""
    
    def __init__(self, data: float, _children: tuple = (), _op: str = ''):
        self.data = data
        self.grad = 0.0  # Gradient
        self._backward = lambda: None  # Backprop function
        self._prev = set(_children)  # Previous nodes in computation graph
        self._op = _op  # Operation that created this node
    
    def __repr__(self):
        return f'Value(data={self.data})'
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f'**{other}')
        
        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out
    
    def __rmul__(self, other):
        return self * other
    
    def __radd__(self, other):
        return self + other
    
    def __sub__(self, other):
        return self + (-other)
    
    def __rsub__(self, other):
        return other + (-self)
    
    def __truediv__(self, other):
        return self * (other ** -1)
    
    def __rtruediv__(self, other):
        return other * (self ** -1)
    
    def __neg__(self):
        return self * -1
    
    def tanh(self):
        """Hyperbolic tangent activation."""
        x = self.data
        t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
        out = Value(t, (self,), 'tanh')
        
        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out
    
    def relu(self):
        """Rectified Linear Unit activation."""
        out = Value(0 if self.data < 0 else self.data, (self,), 'relu')
        
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        """Topological sort and backpropagation."""
        topo = []
        visited = set()
        
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        
        build_topo(self)
        self.grad = 1.0
        
        for v in reversed(topo):
            v._backward()
    
    def zero_grad(self):
        """Reset gradient to zero."""
        self.grad = 0.0

# Test the Value class
a = Value(3.0)
b = Value(2.0)
c = a * b + a ** 2
c.backward()
print(f'c.data = {c.data}, c.grad = {c.grad}')
print(f'a.grad = {a.grad}, b.grad = {b.grad}')

c.data = 15.0, c.grad = 1.0
a.grad = 8.0, b.grad = 3.0


## Part 2: Neural Network Layers

In [8]:
import random

class Neuron:
    """Single neuron with weights and bias."""
    
    def __init__(self, nin: int):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))
    
    def __call__(self, x):
        """Forward pass: activation(w*x + b)."""
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()  # or .relu()
    
    def parameters(self):
        return self.w + [self.b]

class Layer:
    """Fully connected layer."""
    
    def __init__(self, nin: int, nout: int):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        """Forward pass through all neurons."""
        return [n(x) for n in self.neurons]
    
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    """Multi-layer perceptron."""
    
    def __init__(self, nin: int, nouts: list):
        sizes = [nin] + nouts
        self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        """Forward pass through network."""
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for l in self.layers for p in l.parameters()]

# Create a simple 2-layer network
net = MLP(2, [16, 1])
print(f'Network has {len(net.parameters())} parameters')

Network has 65 parameters


## Part 3: Training on Classification Data

In [9]:
# Generate synthetic data: two moons dataset
import math

def generate_data(n=100):
    X, y = [], []
    for i in range(n):
        t = i / n
        # First moon
        if i < n // 2:
            x = math.cos(math.pi * t) + random.gauss(0, 0.1)
            y_val = math.sin(math.pi * t) + random.gauss(0, 0.1)
            label = 1.0
        # Second moon
        else:
            x = 1 - math.cos(math.pi * t) + random.gauss(0, 0.1)
            y_val = 0.5 - math.sin(math.pi * t) + random.gauss(0, 0.1)
            label = -1.0
        X.append([x, y_val])
        y.append(label)
    return X, y

# Generate training data
X, y = generate_data(100)
print(f'Generated {len(X)} training samples')
print(f'Sample: {X[0]} -> {y[0]}')

Generated 100 training samples
Sample: [1.1608081526920826, 0.14744064898166007] -> 1.0


In [ ]:
# Training loop
learning_rate = 0.01
epochs = 100

losses = []

for epoch in range(epochs):
    # Forward pass
    predictions = []
    loss = None
    
    for xi, yi in zip(X, y):
        # Convert to Value objects
        xi_vals = [Value(x) for x in xi]
        yi_val = Value(yi)
        
        # Forward
        pred = net(xi_vals)[0]
        predictions.append(pred)
        
        # MSE loss: (pred - target)^2
        diff = pred - yi_val
        sample_loss = diff * diff
        
        if loss is None:
            loss = sample_loss
        else:
            loss = loss + sample_loss
    
    # Average loss
    loss = loss * (1.0 / len(X))
    losses.append(loss.data)
    
    # Backward pass
    # Zero gradients
    for p in net.parameters():
        p.grad = 0.0
    
    loss.backward()
    
    # Update parameters
    for p in net.parameters():
        p.data -= learning_rate * p.grad
    
    if epoch % 10 == 0:
        print(f'Epoch {epoch}: Loss = {loss.data:.4f}')

print(f'Final loss: {losses[-1]:.4f}')

Epoch 0: Loss = 0.4601
Epoch 10: Loss = 0.2208
Epoch 20: Loss = 0.1555
Epoch 30: Loss = 0.1400
Epoch 40: Loss = 0.1308
Epoch 50: Loss = 0.1231
Epoch 60: Loss = 0.1163
Epoch 70: Loss = 0.1103
Epoch 80: Loss = 0.1049
Epoch 90: Loss = 0.1000
Final loss: 0.0960


In [ ]:
# Evaluate accuracy
correct = 0
for xi, yi in zip(X, y):
    xi_vals = [Value(x) for x in xi]
    pred = net(xi_vals)[0]
    pred_label = 1.0 if pred.data > 0 else -1.0
    if pred_label == yi:
        correct += 1

accuracy = correct / len(X) * 100
print(f'Accuracy: {accuracy:.1f}%')

Accuracy: 99.0%


## Part 4: Understanding Gradients

Let's trace how gradients flow through the network.

In [ ]:
# Trace gradients for a single sample
x_sample = [Value(X[0][0]), Value(X[0][1])]
y_sample = Value(y[0])

# Forward
pred = net(x_sample)[0]
loss = (pred - y_sample) ** 2

# Backward
for p in net.parameters():
    p.grad = 0.0

loss.backward()

# Check gradient magnitudes
grad_norms = []
for layer_idx, layer in enumerate(net.layers):
    layer_grads = [p.grad for p in layer.parameters()]
    norm = sum(g**2 for g in layer_grads) ** 0.5
    grad_norms.append(norm)
    print(f'Layer {layer_idx} gradient norm: {norm:.4f}')

Layer 0 gradient norm: 5.0280
Layer 1 gradient norm: 3.9965


## Key Concepts

1. **Value**: Wraps scalars with gradient tracking
2. **Computation Graph**: Implicit DAG of operations
3. **Backward Pass**: Topological sort + chain rule
4. **Gradient Descent**: Update parameters in direction of -gradient
5. **Backpropagation**: Efficient gradient computation via dynamic programming